# ExpenseLM on Kaggle — finish the pipeline (clean E2 → DPO/E3 → GGUF)

**Before running:** Add your `expenselm` dataset (Add Input, right panel).
Settings → Accelerator: **GPU T4 x2**; Internet: **On**.

Reuses the SFT adapter you already trained (in the dataset), so no 1.5h
retrain. Everything writes to /kaggle/working, which persists as the
notebook's output — download from there when done.

In [ ]:
# 1. setup — locate repo (any nesting), copy to writable dir, install, drop torchao
import os, sys, glob, shutil, subprocess
print("under /kaggle/input:", glob.glob('/kaggle/input/*'))
hits = glob.glob('/kaggle/input/**/src/expenselm/prompts.py', recursive=True)
if hits:
    REPO = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
    if REPO.startswith('/kaggle/input'):
        shutil.copytree(REPO, '/kaggle/working/repo', dirs_exist_ok=True); REPO = '/kaggle/working/repo'
else:
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, ("Nothing under /kaggle/input — click 'Add Input' and attach the expenselm dataset. "
                  "Contents: " + str(glob.glob('/kaggle/input/*')))
    subprocess.run(f'mkdir -p /kaggle/working/repo && unzip -q "{zips[0]}" -d /kaggle/working/repo', shell=True)
    hits = glob.glob('/kaggle/working/repo/**/src/expenselm/prompts.py', recursive=True)
    REPO = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
os.chdir(REPO); sys.path.insert(0, os.path.join(REPO,'src'))
subprocess.run('pip install -q -e . unsloth trl peft bitsandbytes', shell=True)  # -e . = install the CLI
subprocess.run('pip uninstall -y torchao', shell=True)
print('REPO:', REPO, '| adapter present:', os.path.isfile(os.path.join(REPO,'models/sft-adapter/adapter_model.safetensors')))

In [ ]:
# 2. CLEAN E2 — the tool_call bug is already fixed in HFSystem (suppress_tokens)
!python -m expenselm.cli eval --system e2 --split data/splits/test.jsonl
!cp -r results /kaggle/working/results_e2

If E2 schema validity is now ~99% (was 81%), the `<tool_call>` fix worked
and E2 is clean. Now build E3.

In [ ]:
# 3. harvest the SFT model's real mistakes -> DPO preference pairs
!python -m expenselm.train.harvest_failures \
    --splits data/splits/dev.jsonl data/splits/train.jsonl \
    --out dpo_pairs.jsonl --target 500 --adapter models/sft-adapter
!wc -l dpo_pairs.jsonl

In [ ]:
# 4. DPO training (refines the SFT adapter). dpo_unsloth.py expects these paths:
!ln -sfn models/sft-adapter sft-adapter          # script loads 'sft-adapter'
%run src/expenselm/train/dpo_unsloth.py
!mkdir -p models && cp -r dpo-adapter models/dpo-adapter
!cp -r dpo-adapter /kaggle/working/dpo-adapter    # persist the E3 adapter

In [ ]:
# 5. E3 eval (the DPO model)
!python -m expenselm.cli eval --system e3 --split data/splits/test.jsonl
# gate: also check E3 vs E2 on DEV before trusting E3 (PRD risk: DPO can hurt)
!python -m expenselm.cli eval --system e3 --split data/splits/dev.jsonl --limit 100
!cp -r results /kaggle/working/results_e3

In [ ]:
# 6. GGUF export (merge + quantize) for the E5 local demo on your Mac
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained('dpo-adapter', max_seq_length=2048, load_in_4bit=True)
model.save_pretrained_gguf('/kaggle/working/expenselm-gguf', tok, quantization_method='q4_k_m')
print('GGUF written to /kaggle/working/expenselm-gguf — download it for LM Studio')

In [ ]:
# 7. report + list everything to download
!python -m expenselm.cli report
print('\nDOWNLOAD from /kaggle/working:', os.listdir('/kaggle/working'))